In [1]:
from data_gen import make_deliveries
df, late = make_deliveries()

df.columns

Index(['distance', 'prep', 'hour', 'weather', 'vehicle', 'is_rush'], dtype='object')

In [2]:
df

,distance,prep,hour,weather,vehicle,is_rush
0,15.705165,30.850314,21,rain,van,0
1,9.338690,16.100361,21,sunny,car,0
2,17.313360,36.150681,19,snow,bike,1
3,14.249993,25.784069,10,rain,car,0
4,2.789370,9.410922,12,sunny,bike,1
...,...,...,...,...,...,...
795,7.583094,28.846532,13,sunny,van,1
796,1.361649,11.717241,8,rain,van,0
797,4.144267,11.141179,20,sunny,van,1
798,14.778483,33.419587,18,rain,van,1


In [3]:
late

array([1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0,
       1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1,
       1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0,
       0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0,
       0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1,
       0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0,
       0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0,
       0, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0,
       0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0,
       1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1,
       0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1,

In [8]:
from data_gen import make_deliveries
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

df, late = make_deliveries()
pre = ColumnTransformer([
    ("num", StandardScaler(), ["distance", "prep", "hour", "is_rush"]),
    ("cat", OneHotEncoder(handle_unknown="ignore"), ["weather", "vehicle"]),
])
models = {
    "Logistic": Pipeline([("prep", pre), ("clf", LogisticRegression(max_iter=1000))]),
    "Tree d=4": Pipeline([("prep", pre), ("clf", DecisionTreeClassifier(max_depth=4, random_state=42))]),
    "Forest 300": Pipeline([("prep", pre), ("clf", RandomForestClassifier(n_estimators=300, random_state=42))]),
}
for name, m in models.items():
    s = cross_val_score(m, df, late, cv=5)
    print(name, round(s.mean(), 3), round(s.std(), 3))

/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: Runti

Logistic 0.784 0.028
Tree d=4 0.772 0.024
Forest 300 0.79 0.017


In [12]:
pipe  = Pipeline([("prep", pre), ("clf", RandomForestClassifier(n_estimators=300, random_state=42))])

In [17]:
pipe.fit(df, late)

names = pipe.named_steps["prep"].get_feature_names_out()   # the column names
imp   = pipe.named_steps["clf"].feature_importances_        # the numbers
ranked = sorted(zip(names, imp), key=lambda p: p[1], reverse=True)

for name, val in ranked[:6]:
    print(f"  {val:.3f}  {name}")

# imp

  0.352  num__distance
  0.247  num__prep
  0.123  num__hour
  0.088  cat__weather_sunny
  0.054  num__is_rush
  0.038  cat__weather_snow


In [6]:
from data_gen import make_deliveries
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

df, _ = make_deliveries()               # ignore the label — unsupervised
X = df[["distance", "prep", "hour"]]
X_scaled = StandardScaler().fit_transform(X)   # scale first! (Day 17)

for k in range(2, 6):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X_scaled)
    print(k, round(silhouette_score(X_scaled, labels), 3))

2 0.245
3 0.25
4 0.281
5 0.276


/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/kbshal/mero_space/sarathi_academy/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered 